
# PERM disclosure data and recruitment timing, step by step

Introductory notebook for the PERM side of `jollux-immigration-data`:
load a small synthetic PERM fixture, run the public `ingest_perm`
function, and compute the recruitment-start-to-filing interval with
`recruitment_statistics`. No network access, credentials, database,
or production data is needed.

Run from the repository root so the relative fixture path resolves:

```bash
python -m pip install -e .
jupyter notebook notebooks/perm-intro.ipynb
```

See `notebooks/README.md` for the full environment notes.


In [1]:

import csv
from pathlib import Path

from jollux_immigration_data import ingest_perm

fixture = Path("tests/fixtures/perm.csv")

with fixture.open(encoding="utf-8") as handle:
    fixture_rows = sum(1 for _ in csv.reader(handle)) - 1

perm_records = list(ingest_perm(fixture))
print(f"fixture rows: {fixture_rows}")
print(f"records kept after PERM ingestion: {len(perm_records)}")


fixture rows: 2
records kept after PERM ingestion: 2


In [2]:

import json

record = perm_records[0]
selected = {
    key: record[key]
    for key in (
        "case_number",
        "employer_name",
        "employer_normalized",
        "received_date",
        "recruitment_start_date",
        "recruitment_end_date",
    )
}
print(json.dumps(selected, indent=2, ensure_ascii=False))


{
  "case_number": "A-100",
  "employer_name": "Example Analytics Incorporated",
  "employer_normalized": "example analytics",
  "received_date": "2026-04-01",
  "recruitment_start_date": "2026-01-01",
  "recruitment_end_date": "2026-01-31"
}



PERM records also carry the recruitment window
(`recruitment_start_date`, `recruitment_end_date`) alongside the filing
date (`received_date`) and the decision date. The following cell turns
that window into a descriptive interval per record.


In [3]:

import json

from jollux_immigration_data import recruitment_statistics

stats = recruitment_statistics(perm_records)
print(json.dumps(stats, indent=2))


{
  "total_records": 2,
  "included_records": 2,
  "excluded_records": 0,
  "minimum_days": 69,
  "maximum_days": 90,
  "median_days": 79.5
}



**This interval is not official PERM processing time.**
`recruitment_statistics` measures the calendar days between
recruitment start and the PERM filing (`received_date`), derived from
the source records themselves. It does not measure government
adjudication time and should not be presented as a processing-time
estimate. Negative or over-366-day intervals are excluded. See the
[Recruitment statistic](../README.md#recruitment-statistic) section of
the main README for the full caveats.
